# SQD + SBD walkthrough — H2O (serial)

Same self-consistent SQD workflow as `run_sqd_sbd.py`, but interactive and serial.
Runs on a single MPI rank (mpi4py auto-initializes `MPI.COMM_WORLD` with size=1 inside a Jupyter kernel).

Workload: H2O FCIDUMP (NORB=24, NELEC=10), 3000 uniform-random bitstrings, 1 batch, 2 SQD iterations.
Should converge in a few seconds on CPU.

**Reference driver:** [`run_sqd_sbd.py`](./run_sqd_sbd.py) — same recipe, different shape.

## 1. Imports + environment check

In [1]:
from functools import partial
import json, numpy as np
from pathlib import Path
from mpi4py import MPI
from pyscf import ao2mo, tools
from qiskit.primitives import BitArray
from qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian

import sbd
from sbd.sbd_solver import solve_sci_batch
from sbd.device_config import DeviceConfig, print_device_info

comm = MPI.COMM_WORLD
print(f'SBD version            : {sbd.__version__}')
print(f'Compiled SBD backends  : {sbd.available_backends()}')
print(f'MPI library            : {MPI.Get_library_version().splitlines()[0]}')
print(f'MPI world size         : {comm.Get_size()}    (Jupyter kernel = single process)')
print(f'My rank                : {comm.Get_rank()}')
print()
print_device_info()

SBD version            : 1.5.0
Compiled SBD backends  : ['cpu']
MPI library            : Open MPI v5.0.9, package: Open MPI brew@Tahoe-arm64.local Distribution, ident: 5.0.9, repo rev: v5.0.9, Oct 30, 2025 
MPI world size         : 1    (Jupyter kernel = single process)
My rank                : 0

SBD Device Information
✗ No GPU detected
✓ CPU Available: Always


## 2. Load the Hamiltonian (H2O FCIDUMP)

FCIDUMP carries the one- and two-electron integrals plus orbital count and electron count.
PySCF reads it; we extract `hcore` and `eri` for the SQD outer loop.

In [2]:
fcidump_path = '../../data/h2o/fcidump.txt'
norb, nelec_total, ms2 = 24, 10, 0          # h2o constants
num_elec_a = (nelec_total + ms2) // 2        # 5 alpha
num_elec_b = (nelec_total - ms2) // 2        # 5 beta

mf    = tools.fcidump.to_scf(fcidump_path)
hcore = mf.get_hcore()
eri   = ao2mo.restore(1, mf._eri, norb)
nuc   = mf.mol.energy_nuc()
print(f'NORB={norb}, nelec=({num_elec_a},{num_elec_b}), nuc={nuc:.6f}')

Parsing ../../data/h2o/fcidump.txt
NORB=24, nelec=(5,5), nuc=9.193913


## 3. Bitstrings — load hardware-derived counts

Quantum-measurement bitstrings come in as a `BitArray`. The bundled `count_dict_h2o.json` is a `{bitstring: count}` dict from hardware sampling — its strings already satisfy the (5α, 5β) Hamming-weight constraint for h2o.

Uniform random sampling on 48 qubits would *not* work here: the qiskit-addon-sqd configuration-recovery step rejects bitstrings whose Hamming weight doesn't match `nelec`, and random 48-bit strings overwhelmingly fail that check.

In [3]:
def load_counts_as_bitarray(counts_path, num_bits):
    counts = json.loads(Path(counts_path).read_text())
    keys, repeats = list(counts.keys()), list(counts.values())
    flat = np.frombuffer(''.join(keys).encode(), dtype=np.uint8) == ord('1')
    matrix = flat.reshape(len(keys), -1)
    if any(c > 1 for c in repeats):
        matrix = np.repeat(matrix, repeats, axis=0)
    return BitArray.from_bool_array(matrix)

bit_array = load_counts_as_bitarray('count_dict_h2o.json', norb*2)
print(f'Loaded {bit_array.num_shots} bitstrings from count_dict_h2o.json')
print(f'  bits per shot: {norb*2}   (24 spatial × 2 spins)')

Loaded 506 bitstrings from count_dict_h2o.json
  bits per shot: 48   (24 spatial × 2 spins)


## 4. Wire SBD into qiskit-addon-sqd's `sci_solver=` slot

No `sbd.init()` and no `mpi_comm=` needed: `solve_sci_batch` auto-initializes the SBD backend on first call and falls back to `MPI.COMM_WORLD` when `mpi_comm` is omitted (in a Jupyter kernel that's the size-1 communicator).

In [4]:
sbd_config = {
    'method': 0,           # 0 = Davidson
    'eps': 1e-8,           # Davidson convergence tolerance
    'max_it': 100,         # max Davidson iterations per diagonalization
    'max_nb': 50,          # block size
    'do_rdm': 0,           # 0 = density only (sufficient for SQD)
    'do_shuffle': 0,
    'carryover_type': 1,   # singles-only carryover (stable default)
    'ratio': 0.1,
    'threshold': 1e-4,
    'bit_length': 64,
    # serial: 1x1x1 MPI sub-communicator grid
    'adet_comm_size': 1,
    'bdet_comm_size': 1,
    'task_comm_size': 1,
}

sbd_solver = partial(
    solve_sci_batch,
    sbd_config=sbd_config,
    device_config=DeviceConfig.cpu(),       # macOS / no-GPU notebook → cpu
    fcidump_path=fcidump_path,
)

## 5. Run the SQD self-consistent loop

Sample → configuration recovery → subsample → SBD diagonalize → carryover → repeat.
1 batch per iteration, 2 SQD iterations.

In [5]:
rng = np.random.default_rng(42)

result_history = []
def callback(results):
    result_history.append(results)
    iteration = len(result_history)
    for i, r in enumerate(results):
        total_e = r.energy + nuc
        dim = np.prod(r.sci_state.amplitudes.shape)
        print(f'  iter {iteration}, batch {i}:  E = {total_e:.10f} Ha   subspace dim = {dim:_}')

result = diagonalize_fermionic_hamiltonian(
    hcore, eri, bit_array,
    norb=norb,
    nelec=(num_elec_a, num_elec_b),
    samples_per_batch=3000,
    num_batches=1,
    max_iterations=2,
    sci_solver=sbd_solver,
    symmetrize_spin=True,
    callback=callback,
    seed=rng,
)

print()
print(f'Final SQD energy:  {result.energy + nuc:.10f} Ha   (electronic {result.energy:.10f} + nuc {nuc:.6f})')

 Elapsed time for helper construction 0.00089 (sec) 
 Elapsed time for init 1.6e-05 (sec) 
 Elapsed time for makeQChamDiagTerms 0.003941 (sec) 
 Davidson iteration 0.0 (tol=0.0348416): -18.9892
 Davidson iteration 0.1 (tol=0.00104627): -18.9903 -17.8271
 Davidson iteration 0.2 (tol=2.545e-06): -18.9903 -17.828 -16.5233
 Davidson iteration 0.3 (tol=6.63159e-08): -18.9903 -17.9677 -17.8087 -16.5232
 Davidson iteration 0.4 (tol=5.6824e-10): -18.9903 -17.9696 -17.809 -16.5884
 Elapsed time for davidson 0.020181 (sec) 
 Elapsed time for diagonalization 0.020185 (sec) 
 Elapsed time for mult 0.002353 (sec) 
 Energy = -18.99028360239725
 Elapsed time for measurement 0.000204 (sec) 
 truncated weight in carry-over for alpha-det = -2.220446049250313e-16
 truncated weight in carry-over for beta-det = -2.220446049250313e-16
  iter 1, batch 0:  E = -18.9902836024 Ha   subspace dim = 100
 Elapsed time for helper construction 0.000714 (sec) 
 Elapsed time for init 1e-06 (sec) 
 Elapsed time for make

## Notes

- **Why this works in a notebook**: SBD is MPI-native at the C++ level, but a Jupyter kernel is a single Python process. `mpi4py` auto-initializes MPI with `MPI.COMM_WORLD` of size 1; SBD's collectives all become no-ops; the full SQD loop runs on rank 0.
- **Where the equivalent multi-rank version lives**: [`run_sqd_sbd.py`](./run_sqd_sbd.py), which you launch with `mpirun -np N python run_sqd_sbd.py …` for production scale.
- **GPU**: pass `DeviceConfig.gpu()` (Thrust) or `DeviceConfig.gpu_omp()` (LLVM offload) when running on a CUDA-capable machine. Not available on macOS — keep `DeviceConfig.cpu()` here.